# MemoRAG: Memory-Augmented Retrieval over a PDF

A simplified implementation of **MemoRAG**: a RAG approach that adds a *global memory* step before retrieval. Instead of searching the document with the raw question, an LLM-built memory of the whole document is consulted first to write better search queries (*clues*), which are then run against the original document chunks.

**Demo document:** `data/Understanding_Climate_Change.pdf` (33 pages).

## Pipeline
1. **Index the document:** load PDF, split (1000 chars, 200 overlap), embed, store in FAISS (*document vector store*).
2. **Build the memory:** the LLM extracts topic/details pairs from the document; these are embedded and stored in a second FAISS index (*memory store*).
3. **Answer a question:**
   - search the memory store (top 10 entries)
   - the LLM turns them into clues: text spans + surrogate questions
   - each clue (plus the original question) is searched in the document store (top 3 chunks each)
   - deduplicate chunks, then the LLM answers using only those chunks

Paper notation used in comments: `y = Θ_mem(q, D)` (clue generation), `c = Γ(y, D)` (retrieval), `y = Θ(q, c)` (generation).

## Stack
| Component | Choice |
|---|---|
| LLM | Cohere `command-a-03-2025` (`cohere.ClientV2`) |
| Embeddings | `sentence-transformers/all-MiniLM-L6-v2` |
| Vector store | FAISS |
| PDF loading / chunking | `PyPDFLoader` / `RecursiveCharacterTextSplitter` |
| Validation | `pydantic` |

## Setup
```bash
pip install cohere python-dotenv pydantic faiss-cpu langchain-community \
    langchain-huggingface langchain-text-splitters sentence-transformers pypdf
```
Create a `.env` file containing `COHERE_API_KEY=your_key_here`, and place the PDF in `data/`. The memory store is saved to `data/Understanding_Climate_Change_Memory_Store.faiss` (FAISS creates a *directory* at that path). **Run cells top to bottom**: later cells depend on names defined earlier.

## Cell 1: Schemas and prompts
- `KeyValuePair` (`topic`, `details`) and `MemoryResponse` (`pairs`): Pydantic models that validate the LLM's JSON memory output.
- `memory_store_system_prompt` / `memory_store_kv_cache_prompt`: prompts for memory creation (strict JSON output, takes `{document}`).
- `system_prompt_for_queries_creation`: system prompt for clue generation.
- `memorag_span_prompt_for_queries_creation`: asks for **text-span clues** (takes `{question}`, `{relevant_info}`).
- `memorag_sur_prompt_for_queries_creation`: asks for **surrogate questions** (same inputs).

> `ChatGroq`, `pandas` and `sys` are imported but unused; `load_dotenv` is imported here but called in the next cell.

In [13]:
import json
import os
import sys
import pandas as pd
from dotenv import load_dotenv
from pydantic import BaseModel
from typing import List
from langchain_groq import ChatGroq

class KeyValuePair(BaseModel):
    """ The key-value pairs are the "holders" of memory
        The keys are topics from the relevant article and the values are details about that topic """
    topic: str
    details: str

class MemoryResponse(BaseModel):
    """ The obtained memory is a list of topic-details pairs """
    pairs: List[KeyValuePair]
    
memory_store_system_prompt = """
    You are an expert at extracting structured key-value pairs from text.
    "Identify key topics, entities, or questions and pair them with relevant, detailed information.
    "Ensure the output is well-structured, avoiding redundancy while preserving meaning.
    """
    
memory_store_kv_cache_prompt = """
    You are provided with a long article, chunk by chunk. Read each chunk carefully and extract key topics and their detailed information.
    For each key topic:
    1. Identify the main concept, entity, or potential question
    2. Provide the corresponding detailed information or answer

    Note: the aim is to mimic kv cache memory creation

    Now, the article begins:
    {document}

    The article ends here.

    RESPONSE FORMAT IS IMPORTANT. You must return a JSON object with the following structure:
    {{
        "pairs": [
            {{
                "topic": "topic1",
                "details": "details1"
            }},
            {{
                "topic": "topic2",
                "details": "details2"
            }}
        ]
    }}
    """
    
system_prompt_for_queries_creation = "You are an expert in information retrieval. Your task is to extract specific and relevant information from the provided context to help answer the given question."
memorag_span_prompt_for_queries_creation = """
    You are given a question related to an article. To answer it effectively, you need to use specific details from the article. You are not provided with the whole article. Instead, you are provided with specific and relevant information from it.
     Your task is to identify and extract one or more specific clue texts from the provided information that are relevant to the question.

    ### Question: {question}
    ### Information: {relevant_info}
    ### Instructions:
    1. You have a general understanding of the provided information. Your task is to generate one or more specific clues that will help in searching for supporting evidence within the article.
    2. The clues are in the form of text spans that will assist in answering the question.
    3. Only output the clues. If there are multiple clues, separate them with a newline.
    """
    
memorag_sur_prompt_for_queries_creation = """
    You are given a question related to an article. To answer it effectively, you need to use specific details from the article.  You are not provided with the whole article. Instead, you are provided with specific and relevant information from it.
    Your task is to generate precise clue questions that can help locate the necessary information for answering the question.

    ### Question: {question}
    ### Information: {relevant_info}
    ### Instructions:
    1. You have a general understanding of the provided information. Your task is to generate one or more specific clues that will help in searching for supporting evidence within the article.
    2. The clues are in the form of precise surrogate questions that clarify the original question.
    3. Only output the clues. If there are multiple clues, separate them with a newline.
    """

## Cell 2: Cohere connectivity test
Loads `.env`, creates the global client `co` and sends a test prompt. If an explanation prints, your API key and model access work. All later cells use `co`.

In [2]:
import os
import cohere
from dotenv import load_dotenv

load_dotenv()

co = cohere.ClientV2(
    api_key=os.getenv("COHERE_API_KEY")
)

response = co.chat(
    model="command-a-03-2025",
    messages=[
        {
            "role": "user",
            "content": "Explain RAG in simple terms."
        }
    ]
)

print(response.message.content[0].text)

RAG stands for **Retrieval-Augmented Generation**. It’s a technique used in artificial intelligence, particularly with language models like GPT, to improve their ability to provide accurate and relevant answers. Here’s a simple breakdown:

1. **Retrieval**: The model first searches a large database or source of information (like a library or the internet) to find relevant facts or context related to the question.

2. **Augmented**: It combines the retrieved information with its own knowledge (what it has learned during training).

3. **Generation**: Finally, it uses both the retrieved information and its internal knowledge to generate a detailed, accurate, and contextually appropriate response.

Think of it like this:  
- You ask a question.  
- The model acts like a librarian, quickly finding the most relevant books (retrieval).  
- It reads those books to get the facts (augmentation).  
- Then, it writes a clear and informed answer for you (generation).  

This approach helps reduce 

## Cell 3: `MemoryStore` class
Creates the global `embeddings` model and defines the memory module.

**State:** `store` (FAISS index, `None` until first `memorize`), `processed_count`, `json_parse_failures` (warning printed every 5).

| Method | Description |
|---|---|
| `_get_response_text` | Returns `response.message.content[0].text` from a Cohere v2 response. |
| `memorize(document)` | LLM (JSON mode) extracts topic/details pairs, formats as `"Topic: …\nDetails: …"`, adds to FAISS. Later calls skip pairs whose topic already exists. |
| `_parse_into_pairs` | Parses and validates JSON via `MemoryResponse`; falls back to the text parser on failure. |
| `_parse_text_into_pairs` | Fallback parser for `Topic:` / `Details:` lines. |
| `create_retrieval_queries(query)` | Core MemoRAG step (below). Returns `None` if the store is empty. |
| `save_store` / `load_store` | Persist/load with `FAISS.save_local` / `load_local`. |

**`create_retrieval_queries` steps:** (1) top `min(10, ntotal)` memory entries for the query → (2) LLM call for text-span clues → (3) LLM call (`temperature=0.5`) for surrogate questions → (4) split on newlines, keep lines of more than 3 words → (5) append the original query → return.

> `load_store` uses `allow_dangerous_deserialization=True` (FAISS uses pickle). Only load files you created yourself.

In [15]:
import json

from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS


embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)


class MemoryStore:
    """
    MemoryStore implements the memory module used in MemoRAG-style retrieval.

    It:
    1. Takes a document/chunk.
    2. Uses an LLM to compress it into topic/details pairs.
    3. Embeds those compressed memories.
    4. Stores them in FAISS.
    5. Uses the memory store later to generate retrieval queries.
    """

    def __init__(self):
        # embeddings is already instantiated above
        self.embeddings = embeddings

        self.store = None
        self.processed_count = 0

        self.json_parse_failures = 0
        self._last_parse_used_fallback = False

    # ---------------------------------------------------------
    # Helper: extract text from Cohere response
    # ---------------------------------------------------------

    def _get_response_text(self, response):
        """
        Extract generated text from Cohere v2 response.

        Cohere v2:
            response.message.content[0].text
        """

        try:
            return response.message.content[0].text

        except (AttributeError, IndexError, TypeError) as e:
            raise ValueError(
                f"Could not extract text from Cohere response.\n"
                f"Response type: {type(response)}\n"
                f"Response: {response}"
            ) from e

    # ---------------------------------------------------------
    # Memorize
    # ---------------------------------------------------------

    def memorize(self, document: str):
        """
        Process document into topic/details pairs
        and store them in FAISS.
        """

        system_prompt = memory_store_system_prompt
        kv_cache_prompt = memory_store_kv_cache_prompt

        print(f"Processing chunk {self.processed_count + 1}...")

        response = co.chat(
            model="command-a-03-2025",
            messages=[
                {
                    "role": "system",
                    "content": system_prompt
                },
                {
                    "role": "user",
                    "content": kv_cache_prompt.format(
                        document=document
                    )
                }
            ],
            response_format={
                "type": "json_object"
            }
        )

        # FIX:
        # Cohere v2 does NOT use response.choices
        content = self._get_response_text(response)

        pairs = self._parse_into_pairs(content)

        if self._last_parse_used_fallback:
            self.json_parse_failures += 1

            if (
                self.json_parse_failures > 0
                and self.json_parse_failures % 5 == 0
            ):
                print(
                    f"Warning: JSON parsing has failed "
                    f"{self.json_parse_failures} times"
                )

        print("*" * 20)

        print(
            f"First extracted pairs: "
            f"{pairs[0:3]}\n"
        )

        # -----------------------------------------------------
        # Convert pairs into documents for FAISS
        # -----------------------------------------------------

        texts = []
        metadatas = []

        for topic, details in pairs:

            if not topic or not details:
                continue

            combined_text = (
                f"Topic: {topic}\n"
                f"Details: {details}"
            )

            texts.append(combined_text)

            metadatas.append(
                {
                    "topic": topic
                }
            )

        print("*" * 20)

        print(
            f"Num of extracted texts: {len(texts)}\n"
        )

        # Nothing useful was extracted
        if not texts:
            print("No valid memory pairs were extracted.")
            return

        # -----------------------------------------------------
        # Create FAISS store
        # -----------------------------------------------------

        if self.store is None:

            self.store = FAISS.from_texts(
                texts,
                self.embeddings,
                metadatas=metadatas
            )

        else:

            # Get existing documents
            existing_docs = self.store.similarity_search(
                "",
                k=self.store.index.ntotal
            )

            existing_topics = {
                doc.metadata.get("topic")
                for doc in existing_docs
            }

            # Filter duplicates
            new_texts = []
            new_metadatas = []

            for text, metadata in zip(
                texts,
                metadatas
            ):

                topic = metadata["topic"]

                if topic not in existing_topics:

                    new_texts.append(text)
                    new_metadatas.append(metadata)

            # Add only new memories
            if new_texts:

                self.store.add_texts(
                    new_texts,
                    metadatas=new_metadatas
                )

                print(
                    f"Added {len(new_texts)} new memories."
                )

            else:

                print("No new memories to add.")

        self.processed_count += 1

        print(
            f"Processed {self.processed_count} chunks so far\n"
        )

    # ---------------------------------------------------------
    # Parse JSON
    # ---------------------------------------------------------

    def _parse_into_pairs(self, content: str):
        """
        Parse LLM JSON response into
        [(topic, details), ...]
        """

        self._last_parse_used_fallback = False

        try:

            data = json.loads(content)

            validated_data = MemoryResponse(
                **data
            )

            return [
                (pair.topic, pair.details)
                for pair in validated_data.pairs
            ]

        except (
            json.JSONDecodeError,
            ValueError
        ) as e:

            print(
                f"Error parsing JSON response: {e}"
            )

            self._last_parse_used_fallback = True

            return self._parse_text_into_pairs(
                content
            )

    # ---------------------------------------------------------
    # Fallback text parser
    # ---------------------------------------------------------

    def _parse_text_into_pairs(self, text: str):
        """
        Fallback parser if the LLM does not
        return valid JSON.
        """

        pairs = []

        lines = text.split("\n")

        current_topic = None
        current_details = []

        for line in lines:

            line = line.strip()

            if line.startswith("Topic:"):

                # Save previous pair
                if current_topic:

                    pairs.append(
                        (
                            current_topic,
                            " ".join(current_details)
                        )
                    )

                current_topic = line[6:].strip()
                current_details = []

            elif line.startswith("Details:"):

                current_details.append(
                    line[8:].strip()
                )

            elif current_details:

                current_details.append(
                    line
                )

        # Save last pair
        if current_topic:

            pairs.append(
                (
                    current_topic,
                    " ".join(current_details)
                )
            )

        return pairs

    # ---------------------------------------------------------
    # Create retrieval queries
    # ---------------------------------------------------------

    def create_retrieval_queries(
        self,
        query: str
    ) -> list[str] | None:

        """
        Use the Memory Store to generate
        retrieval queries for the original
        document store.
        """

        if not self.store:

            return None

        # -----------------------------------------------------
        # Search Memory Store
        # -----------------------------------------------------

        k = min(
            10,
            self.store.index.ntotal
        )

        results = (
            self.store.similarity_search_with_score(
                query,
                k=k
            )
        )

        relevant_info = [
            (
                f"Topic: {doc.metadata['topic']}\n"
                f"Details: {doc.page_content}"
            )

            for doc, score in results
        ]

        # -----------------------------------------------------
        # Generate memory-based retrieval clues
        # -----------------------------------------------------

        memorag_span_prompt = (
            memorag_span_prompt_for_queries_creation
        )

        memorag_sur_prompt = (
            memorag_sur_prompt_for_queries_creation
        )

        system_prompt = (
            system_prompt_for_queries_creation
        )

        # -----------------------------------------------------
        # Generate text spans
        # -----------------------------------------------------

        text_spans_response = co.chat(
            model="command-a-03-2025",
            messages=[
                {
                    "role": "system",
                    "content": system_prompt
                },
                {
                    "role": "user",
                    "content": memorag_span_prompt.format(
                        question=query,
                        relevant_info=relevant_info
                    )
                }
            ]
        )

        text_spans = self._get_response_text(
            text_spans_response
        )

        # -----------------------------------------------------
        # Generate surrogate queries
        # -----------------------------------------------------

        surrogate_queries_response = co.chat(
            model="command-a-03-2025",
            messages=[
                {
                    "role": "system",
                    "content": system_prompt
                },
                {
                    "role": "user",
                    "content": memorag_sur_prompt.format(
                        question=query,
                        relevant_info=relevant_info
                    )
                }
            ],
            temperature=0.5
        )

        surrogate_queries = self._get_response_text(
            surrogate_queries_response
        )

        # -----------------------------------------------------
        # Combine generated queries
        # -----------------------------------------------------

        retrieval_query = (
            text_spans.split("\n")
            + surrogate_queries.split("\n")
        )

        retrieval_query = [
            q.strip()
            for q in retrieval_query
            if len(q.split()) > 3
        ]

        # Always include original query
        retrieval_query.append(query)

        return retrieval_query

    # ---------------------------------------------------------
    # Save
    # ---------------------------------------------------------

    def save_store(self, path: str):

        if self.store:

            self.store.save_local(path)

            print(
                f"Memory store saved to: {path}"
            )

        else:

            print("Memory store is empty.")

    # ---------------------------------------------------------
    # Load
    # ---------------------------------------------------------

    def load_store(self, path: str):

        self.store = FAISS.load_local(
            path,
            self.embeddings,
            allow_dangerous_deserialization=True
        )

        print(
            f"Memory store loaded from: {path}"
        )

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 1923.44it/s]


## Cell 4: Retrieval pipeline and demo run
| Function | Description |
|---|---|
| `encode_pdf(path, chunk_size=1000, chunk_overlap=200)` | Load, split, embed, return FAISS document store. |
| `retrieve_context(query, vectorstore, k=3)` | Top-`k` chunk texts for one query. |
| `generate_answer(query, contexts, temperature=0)` | Answers using only the retrieved context; returns a fixed "not available" message if no context. |
| `process_query(query, memory_store, vectorstore)` | Clues → retrieve per clue → deduplicate → answer. Falls back to the raw query if no clues. Returns `(unique_contexts, final_answer)`. |

**Driver:** builds the document store, builds the memory store only if its path doesn't exist (delete the directory to rebuild), loads it, then runs three test questions (biodiversity, summary, social/economic influence).

**Example output:** 33 pages → 97 chunks; memory build yielded 42 entries; the biodiversity question produced 8 retrieval queries, retrieving 24 chunks (11 unique).

**Tunable parameters:** `chunk_size`/`chunk_overlap` (1000/200), memory `k` (10), document `k` (3), clue length filter (>3 words), temperatures (0.5 clues / 0 answer), model name (hardcoded).

### Known issues / suggested improvements
1. The whole PDF is memorized in **one** LLM call (despite the "chunk by chunk" prompt); long documents may exceed context. Memorize per segment instead.
2. Raw `Topic:` / `Details:` lines leak into retrieval queries because `relevant_info` is passed as a Python list. Join with newlines and strip prefixes.
3. Duplicate check is topic-name only and fetches the entire store via `similarity_search("")`. Track a set of topics or compare embeddings.
4. Unsafe deserialization flag (see Cell 3).
5. Unused imports (`ChatGroq`, `pandas`, `sys`).
6. Hardcoded model name; no retries or error handling around API calls.
7. Fallback parser only handles `Topic:`/`Details:` lines.
8. This is a text-based vector memory, not a real KV cache as in the original paper.
9. No evaluation against plain RAG.

In [16]:
import os
from typing import List

from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import FAISS


# ============================================================
# 1. Encode PDF
# ============================================================

def encode_pdf(
    path: str,
    chunk_size: int = 1000,
    chunk_overlap: int = 200
):
    """
    Load a PDF, split it into chunks, embed the chunks,
    and create a FAISS vector store.
    """

    # Load PDF
    loader = PyPDFLoader(path)
    documents = loader.load()

    # Split into chunks
    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=chunk_overlap,
        length_function=len
    )

    texts = text_splitter.split_documents(documents)

    print(f"Loaded {len(documents)} PDF pages")
    print(f"Created {len(texts)} document chunks")

    # Create vector store using the global
    # HuggingFaceEmbeddings object
    vectorstore = FAISS.from_documents(
        texts,
        embeddings
    )

    return vectorstore


# ============================================================
# 2. Retrieve context
# ============================================================

def retrieve_context(
    retrieval_query: str,
    vectorstore,
    k: int = 3
) -> List[str]:

    """
    Retrieve relevant document chunks from the
    original document vector store.
    """

    results = vectorstore.similarity_search(
        retrieval_query,
        k=k
    )

    contexts = [
        doc.page_content
        for doc in results
    ]

    return contexts


# ============================================================
# 3. Generate final answer
# ============================================================

def generate_answer(
    query: str,
    contexts: List[str],
    temperature: float = 0
) -> str:

    """
    Generate the final answer using only
    the retrieved document context.
    """

    if not contexts:
        return (
            "The necessary information is not available "
            "in the retrieved context."
        )

    prompt = f"""
Based ONLY on the provided context, answer the query.

Query:
{query}

Context:
{"\n\n".join(contexts)}

Provide a clear and concise answer based solely on the
information in the context.

If the context does not contain sufficient information
to answer the query, clearly state that the necessary
information is not available in the provided context.
Do not use external knowledge.
"""

    response = co.chat(
        model="command-a-03-2025",
        messages=[
            {
                "role": "system",
                "content": (
                    "You are an expert in answering questions "
                    "from provided documents. "
                    "Use only the supplied context."
                )
            },
            {
                "role": "user",
                "content": prompt
            }
        ],
        temperature=temperature
    )

    # Cohere v2 response format
    return response.message.content[0].text


# ============================================================
# 4. Process query
# ============================================================

def process_query(
    query: str,
    memory_store,
    vectorstore
):

    print("*" * 20)
    print(f"\nProcessing Query: {query}")
    print("=" * 50)

    # --------------------------------------------------------
    # STEP 1:
    # Check Memory Store
    #
    # y = Θ_mem(q, D | θ_mem)
    # --------------------------------------------------------

    retrieval_queries = (
        memory_store.create_retrieval_queries(query)
    )

    print("*" * 20)

    if not retrieval_queries:

        print(
            "\nNo memory-based retrieval queries generated."
        )

        # Fallback:
        # Search the original vector store directly
        retrieval_queries = [query]

    print(
        f"\nRetrieval Queries:\n"
        f"{retrieval_queries}\n"
    )

    # --------------------------------------------------------
    # STEP 2:
    # Search ORIGINAL document vector store
    #
    # c = Γ(y, D | γ)
    # --------------------------------------------------------

    all_contexts = []

    for retrieval_query in retrieval_queries:

        contexts = retrieve_context(
            retrieval_query,
            vectorstore,
            k=3
        )

        all_contexts.extend(contexts)

    # --------------------------------------------------------
    # STEP 3:
    # Remove duplicate chunks
    # --------------------------------------------------------

    unique_contexts = list(
        dict.fromkeys(all_contexts)
    )

    print("*" * 20)

    print(
        f"Retrieved {len(all_contexts)} chunks"
    )

    print(
        f"Unique contexts: {len(unique_contexts)}"
    )

    if unique_contexts:

        print(
            "\nRetrieved Context Example:"
        )

        print(
            unique_contexts[0][:1000]
        )

    else:

        print(
            "\nNo relevant context was retrieved."
        )

    # --------------------------------------------------------
    # STEP 4:
    # Generate final answer
    #
    # y = Θ(q, c | θ)
    # --------------------------------------------------------

    final_answer = generate_answer(
        query,
        unique_contexts
    )

    print("*" * 20)

    print(
        f"\nFinal Answer:\n{final_answer}"
    )

    return unique_contexts, final_answer


# ============================================================
# 5. Paths
# ============================================================

path = r"data/Understanding_Climate_Change.pdf"

memory_store_path = (
    "data/Understanding_Climate_Change_Memory_Store.faiss"
)


# ============================================================
# 6. Create original document vector store
# ============================================================

print("\nCreating document vector store...")

climate_vectorstore = encode_pdf(
    path,
    chunk_size=1000,
    chunk_overlap=200
)


# ============================================================
# 7. Initialize Memory Store
# ============================================================

climate_memory_store = MemoryStore()


# ============================================================
# 8. Build Memory Store ONLY if it doesn't exist
# ============================================================

if not os.path.exists(memory_store_path):

    print("\nMemory store does not exist.")
    print("Creating memory store...")

    loader = PyPDFLoader(path)

    documents = loader.load()

    document_text = "\n".join(
        doc.page_content
        for doc in documents
    )

    climate_memory_store.memorize(
        document_text
    )

    climate_memory_store.save_store(
        memory_store_path
    )

else:

    print("\nMemory store already exists.")
    print("Loading existing memory store...")


# ============================================================
# 9. Load Memory Store
# ============================================================

climate_memory_store.load_store(
    memory_store_path
)


# ============================================================
# 10. Test queries
# ============================================================

query_1 = (
    "What are the impacts of climate change on biodiversity?"
)

query_2 = (
    "Please summarize the climate change article"
)

query_3 = (
    "Describe the social and economic influence of climate change."
)


for query in [
    query_1,
    query_2,
    query_3
]:

    process_query(
        query,
        climate_memory_store,
        climate_vectorstore
    )


Creating document vector store...
Loaded 33 PDF pages
Created 97 document chunks

Memory store does not exist.
Creating memory store...
Processing chunk 1...
********************
First extracted pairs: [('Climate Change Definition', "Climate change refers to significant, long-term changes in the global climate, encompassing the planet's overall weather patterns, including temperature, precipitation, and wind patterns, over an extended period."), ('Historical Context', "The Earth's climate has historically changed due to natural factors like variations in Earth's orbit, leading to cycles of glacial advance and retreat. The Holocene epoch, following the last ice age, marked the beginning of modern climate conditions and human civilization."), ('Modern Observations', 'Modern scientific observations show rapid increases in global temperatures, sea levels, and extreme weather events, primarily driven by human activities, especially greenhouse gas emissions.')]

********************
Num of 